In [100]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

RAW_PATH     = "/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/raw/raw.csv"
OUT_WIDE     = "/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/processed/currency_quarterly.csv"
OUT_LONG     = "/home/dania/SaudiFinHub/Saudi_FinHub_Project/data/processed/currency_quarterly_long.csv"

PERIOD_START = pd.Timestamp("2013-01-01")
PERIOD_END   = pd.Timestamp("2026-03-31")   # inclusive

QUOTES       = ["USD", "EUR", "GBP", "CNY", "AED"]

In [101]:
fx_daily = pd.read_csv(
    RAW_PATH,
    usecols=["date", "base", "quote", "rate"],
    parse_dates=["date"],
    dtype={"base": "string", "quote": "string", "rate": "float64"},
)

fx_daily["base"]  = fx_daily["base"].str.strip().str.upper()
fx_daily["quote"] = fx_daily["quote"].str.strip().str.upper()

print(f"raw rows     : {len(fx_daily):,}")
print(f"date range   : {fx_daily['date'].min().date()} -> {fx_daily['date'].max().date()}")
print(f"base values  : {sorted(fx_daily['base'].unique())}")
print(f"quote values : {sorted(fx_daily['quote'].unique())}")

raw rows     : 25,005
date range   : 2013-01-01 -> 2026-09-10
base values  : ['SAR']
quote values : ['AED', 'CNY', 'EUR', 'GBP', 'USD']


In [102]:
before = len(fx_daily)

fx_daily = (
    fx_daily
    .dropna(subset=["date", "base", "quote", "rate"])
    .query("rate > 0")
    .drop_duplicates(subset=["date", "base", "quote"], keep="last")
    .sort_values(["quote", "date"])
    .reset_index(drop=True)
)

print(f"kept {len(fx_daily):,} / {before:,} daily rows")
print(f"dupes remaining: {fx_daily.duplicated(subset=['date','base','quote']).sum()}")

kept 25,005 / 25,005 daily rows
dupes remaining: 0


In [103]:
fx_daily = fx_daily.loc[
    (fx_daily["date"] >= PERIOD_START) &
    (fx_daily["date"] <= PERIOD_END)
].copy()

fx_daily["year"]         = fx_daily["date"].dt.year.astype("int16")
fx_daily["quarter"]      = "Q" + fx_daily["date"].dt.quarter.astype(str)   # Q1..Q4
fx_daily["year_quarter"] = (                                                # 2013-Q1
    fx_daily["year"].astype(str) + "-" + fx_daily["quarter"]
)

print(f"rows in period         : {len(fx_daily):,}")
print(f"distinct year_quarter  : {fx_daily['year_quarter'].nunique()}")
print(f"first / last           : {fx_daily['year_quarter'].min()} / {fx_daily['year_quarter'].max()}")
print(fx_daily[["date", "year", "quarter", "year_quarter", "quote", "rate"]].head(5).to_string(index=False))

rows in period         : 24,190
distinct year_quarter  : 53
first / last           : 2013-Q1 / 2026-Q1
      date  year quarter year_quarter quote    rate
2013-01-01  2013      Q1      2013-Q1   AED 0.97933
2013-01-02  2013      Q1      2013-Q1   AED 0.97933
2013-01-03  2013      Q1      2013-Q1   AED 0.97933
2013-01-04  2013      Q1      2013-Q1   AED 0.97933
2013-01-05  2013      Q1      2013-Q1   AED 0.97933


In [104]:
fx_quarterly_long = (
    fx_daily
    .groupby(["year", "quarter", "year_quarter", "quote"], as_index=False)
    .agg(
        n_days    = ("rate", "size"),
        rate_mean = ("rate", "mean"),
        rate_std  = ("rate", "std"),
        rate_min  = ("rate", "min"),
        rate_max  = ("rate", "max"),
    )
    .sort_values(["year", "quarter", "quote"])
    .reset_index(drop=True)
)

print(f"quarterly long rows: {len(fx_quarterly_long):,}")
print(fx_quarterly_long.head(10).to_string(index=False))

quarterly long rows: 265
 year quarter year_quarter quote  n_days  rate_mean  rate_std  rate_min  rate_max
 2013      Q1      2013-Q1   AED      90   0.979330  0.000000   0.97933   0.97933
 2013      Q1      2013-Q1   CNY      90   1.660591  0.002161   1.65730   1.66550
 2013      Q1      2013-Q1   EUR      90   0.202173  0.003270   0.19617   0.20834
 2013      Q1      2013-Q1   GBP      90   0.171756  0.004460   0.16418   0.17898
 2013      Q1      2013-Q1   USD      90   0.266670  0.000000   0.26667   0.26667
 2013      Q2      2013-Q2   AED      91   0.979330  0.000000   0.97933   0.97933
 2013      Q2      2013-Q2   CNY      91   1.642090  0.006868   1.63350   1.65760
 2013      Q2      2013-Q2   EUR      91   0.204328  0.002072   0.19938   0.20814
 2013      Q2      2013-Q2   GBP      91   0.173702  0.001856   0.17008   0.17690
 2013      Q2      2013-Q2   USD      91   0.266670  0.000000   0.26667   0.26667


In [105]:
fx_quarterly = (
    fx_quarterly_long
    # raw rate is SAR -> X (e.g. 1 SAR = 0.26667 USD).
    # usd_sar should be "1 USD = ? SAR", so invert.
    .assign(rate_sar_per_unit = lambda d: 1.0 / d["rate_mean"])
    .pivot(
        index=["year", "quarter", "year_quarter"],
        columns="quote",
        values="rate_sar_per_unit",
    )
    .rename(columns={
        "USD": "usd_sar",
        "EUR": "eur_sar",
        "GBP": "gbp_sar",
        "CNY": "cny_sar",
        "AED": "aed_sar",
    })
    .reset_index()
    # enforce exact column order
    .loc[:, ["year", "quarter", "year_quarter",
             "usd_sar", "eur_sar", "gbp_sar", "cny_sar", "aed_sar"]]
    .sort_values(["year", "quarter"])
    .reset_index(drop=True)
)

print(f"quarterly wide rows: {len(fx_quarterly):,}")
print(fx_quarterly.head(10).to_string(index=False))

quarterly wide rows: 53
 year quarter year_quarter  usd_sar  eur_sar  gbp_sar  cny_sar  aed_sar
 2013      Q1      2013-Q1 3.749953 4.946248 5.822228 0.602195 1.021106
 2013      Q2      2013-Q2 3.749953 4.894099 5.756983 0.608980 1.021106
 2013      Q3      2013-Q3 3.749953 4.963242 5.809276 0.611963 1.021106
 2013      Q4      2013-Q4 3.749953 5.102148 6.069266 0.615396 1.021106
 2014      Q1      2014-Q1 3.749953 5.135357 6.203282 0.614673 1.021106
 2014      Q2      2014-Q2 3.749953 5.143463 6.307215 0.602165 1.021106
 2014      Q3      2014-Q3 3.749953 4.971906 6.261847 0.608229 1.021106
 2014      Q4      2014-Q4 3.749953 4.685021 5.936901 0.609943 1.021106
 2015      Q1      2015-Q1 3.749953 4.229135 5.685558 0.601789 1.021106
 2015      Q2      2015-Q2 3.749953 4.143257 5.742383 0.604962 1.021106


In [106]:
# --- T-Q1: unique (year, quarter) ---
assert not fx_quarterly.duplicated(["year", "quarter"]).any(), \
    "duplicate (year, quarter) rows"
print("T-Q1 OK  one row per (year, quarter)")

# --- T-Q2: 53 quarters, no gaps, 2013-Q1 .. 2026-Q1 ---
expected = pd.period_range("2013Q1", "2026Q1", freq="Q").astype(str)
got = pd.PeriodIndex(
    fx_quarterly["year"].astype(str) + fx_quarterly["quarter"],
    freq="Q",
).astype(str).tolist()
assert got == list(expected), (
    f"quarter sequence mismatch\n"
    f"first 5 expected={list(expected)[:5]}  got={got[:5]}\n"
    f"last  5 expected={list(expected)[-5:]}  got={got[-5:]}"
)
print(f"T-Q2 OK  53 quarters, no gaps ({got[0]} -> {got[-1]})")

# --- T-Q3: no nulls in output ---
currency_cols = ["usd_sar", "eur_sar", "gbp_sar", "cny_sar", "aed_sar"]
assert fx_quarterly[currency_cols].notna().all().all(), "null values in output"
print("T-Q3 OK  no nulls")

# --- T-Q4: positive rates, plausible bounds ---
assert (fx_quarterly[currency_cols] > 0).all().all()
assert (fx_quarterly[currency_cols] < 100).all().all(), "rate outside plausible range"
print("T-Q4 OK  all rates positive and < 100")

# --- T-Q5: direction check — EUR/SAR and GBP/SAR must be > 1 ---
assert (fx_quarterly["eur_sar"] > 1).all(), "eur_sar in the wrong direction"
assert (fx_quarterly["gbp_sar"] > 1).all(), "gbp_sar in the wrong direction"
print("T-Q5 OK  eur_sar, gbp_sar in SAR-per-unit direction")

# --- T-Q6: AED/SAR pinned near 0.98 (pegged); USD/SAR pinned near 3.75 ---
assert fx_quarterly["aed_sar"].std() < 0.01, \
    f"AED/SAR moved a lot: std={fx_quarterly['aed_sar'].std()}"
assert fx_quarterly["usd_sar"].std() < 0.01, \
    f"USD/SAR moved a lot: std={fx_quarterly['usd_sar'].std()}"
print("T-Q6 OK  pegged pairs are flat")

# --- T-Q7: spot check against manually computed mean ---
q_year, q_num, c = 2015, "Q2", "EUR"
manual = fx_daily.loc[
    (fx_daily["year"] == q_year)
    & (fx_daily["quarter"] == q_num)
    & (fx_daily["quote"] == c),
    "rate",
].mean()
expected_sar_per_eur = 1.0 / manual
actual = fx_quarterly.loc[
    (fx_quarterly["year"] == q_year) & (fx_quarterly["quarter"] == q_num),
    "eur_sar",
].iloc[0]
np.testing.assert_allclose(actual, expected_sar_per_eur, rtol=1e-9)
print(f"T-Q7 OK  spot check {q_year}-{q_num} {c}: {actual:.5f}")

# --- T-Q8: long form has 5 quotes per quarter ---
assert (fx_quarterly_long.groupby(["year", "quarter"]).size() == 5).all()
print("T-Q8 OK  long form: 5 quotes per quarter")

# --- T-Q9: column order matches spec ---
assert list(fx_quarterly.columns) == [
    "year", "quarter", "year_quarter",
    "usd_sar", "eur_sar", "gbp_sar", "cny_sar", "aed_sar",
]
print("T-Q9 OK  column order as specified")

print("\nAll currency transformation tests passed.")

T-Q1 OK  one row per (year, quarter)
T-Q2 OK  53 quarters, no gaps (2013Q1 -> 2026Q1)
T-Q3 OK  no nulls
T-Q4 OK  all rates positive and < 100
T-Q5 OK  eur_sar, gbp_sar in SAR-per-unit direction
T-Q6 OK  pegged pairs are flat
T-Q7 OK  spot check 2015-Q2 EUR: 4.14326
T-Q8 OK  long form: 5 quotes per quarter
T-Q9 OK  column order as specified

All currency transformation tests passed.


In [107]:
fx_quarterly.to_csv(OUT_WIDE, index=False)
fx_quarterly_long.to_csv(OUT_LONG, index=False)

from pathlib import Path
for p in (OUT_WIDE, OUT_LONG):
    path = Path(p).resolve()
    print(f"saved: {path}  ({path.stat().st_size / 1024:.1f} KB)")

print()
print("wide — first 8 rows:")
print(fx_quarterly.head(8).to_string(index=False))
print()
print("wide — last 8 rows:")
print(fx_quarterly.tail(8).to_string(index=False))

saved: /home/dania/SaudiFinHub/Saudi_FinHub_Project/data/processed/currency_quarterly.csv  (5.7 KB)
saved: /home/dania/SaudiFinHub/Saudi_FinHub_Project/data/processed/currency_quarterly_long.csv  (17.7 KB)

wide — first 8 rows:
 year quarter year_quarter  usd_sar  eur_sar  gbp_sar  cny_sar  aed_sar
 2013      Q1      2013-Q1 3.749953 4.946248 5.822228 0.602195 1.021106
 2013      Q2      2013-Q2 3.749953 4.894099 5.756983 0.608980 1.021106
 2013      Q3      2013-Q3 3.749953 4.963242 5.809276 0.611963 1.021106
 2013      Q4      2013-Q4 3.749953 5.102148 6.069266 0.615396 1.021106
 2014      Q1      2014-Q1 3.749953 5.135357 6.203282 0.614673 1.021106
 2014      Q2      2014-Q2 3.749953 5.143463 6.307215 0.602165 1.021106
 2014      Q3      2014-Q3 3.749953 4.971906 6.261847 0.608229 1.021106
 2014      Q4      2014-Q4 3.749953 4.685021 5.936901 0.609943 1.021106

wide — last 8 rows:
 year quarter year_quarter  usd_sar  eur_sar  gbp_sar  cny_sar  aed_sar
 2024      Q2      2024-Q2 3.74